# Análise Exploratória de Dados (EDA)
## Análise de Dados de Compliance Público - TCC MBA

**Objetivo:** Explorar os datasets da camada Gold no **nível municipal** (5.570 registros, 1 linha por município brasileiro) para entender:
- Distribuições de dados e estatísticas descritivas
- Valores ausentes e qualidade dos dados
- Padrões e relações iniciais
- Variações regionais em indicadores de compliance e socioeconômicos

**Nota sobre granularidade.** O dataset analítico principal usado aqui é `analysis_compliance_municipality` / `analise_compliance_municipio` (uma linha por município, N=5.570). O rollup estadual anterior (`analysis_compliance`, N=27) tinha poucas observações para análise de correlação/regressão significativa. Para preservar o contexto geográfico, `state_code` / `state_name` / `region_code` / `region_name` (e suas contrapartes em pt-BR) são mantidos como colunas identificadoras e codificados one-hot em variáveis dummy (`is_region_*`, `is_state_*`).

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Instala todas as dependências do projeto na primeira execução (Colab, ambientes novos, etc).
# Idempotente: pip ignora o que já está instalado.
# Para regerar esta célula, execute: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Instalando dependências de {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependências prontas.")
else:
    print("requirements.txt não encontrado. Instale manualmente: pip install -r requirements.txt")


## Passo a passo

1. Pacotes e configuração do ambiente
2. Reprodutibilidade
3. Carregamento de dados
4. Blocos de análise
5. Resumo e interpretação


# Pacotes


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Tenta o loader local primeiro; recorre ao S3 se disponível
from src.analysis.pt_br_loader import GoldDataLoaderPtBr as GoldDataLoader

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)
pd.set_option('display.float_format', '{:.2f}'.format)


In [ ]:
import matplotlib as mpl
mpl.rcParams['axes.formatter.useoffset'] = False
mpl.rcParams['axes.formatter.limits'] = (-99, 99)


# Reprodutibilidade


In [ ]:
import os
import random

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Seed de reprodutibilidade fixada em {SEED}")


In [ ]:
import json as _json

_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}

S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))
AWS_PROFILE = os.environ.get('AWS_PROFILE', _rtcfg.get('aws', {}).get('profile', None))


## 1. Carregar Dados

In [ ]:
loader = GoldDataLoader()

print("Datasets disponíveis:")
for dataset in loader.list_available_datasets():
    print(f"  • {dataset}")


In [ ]:
datasets = loader.load_all()

df_muni = datasets.get('municipio_socioeconomico')
df_state = datasets.get('resumo_estado')
df_sanctions = datasets.get('resumo_sancoes')

# Primary analytical dataset: one row per municipality (N ~= 5,570).
df_analysis = datasets.get('analise_compliance_municipio')

# --- Add one-hot region and dummies de estado (not present in the muni dataset). ---
# Espelham as colunas is_norte / is_nordeste / ... que existiam no
# dataset no nível estadual -- mantemos os mesmos nomes legíveis para que o
# código a jusante que referencia essas colunas continue funcionando.
# Mantidas como Int64 (0/1) por consistência com a convenção is_* existente e
# para uso direto como features de regressão nos notebooks 02 / 03.
REGION_NAME_TO_DUMMY = {
    'Norte': 'is_norte',
    'Nordeste': 'is_nordeste',
    'Sudeste': 'is_sudeste',
    'Sul': 'is_sul',
    'Centro-Oeste': 'is_centro_oeste',
}
for rname, col in REGION_NAME_TO_DUMMY.items():
    df_analysis[col] = (df_analysis['nome_regiao'] == rname).astype('Int64')
REGION_DUMMY_COLS = list(REGION_NAME_TO_DUMMY.values())

# Estado dummies: one per state, named by 2-digit IBGE state code.
state_dummies = pd.get_dummies(df_analysis['codigo_estado'], prefix='is_state').astype('Int64')
df_analysis = pd.concat([df_analysis, state_dummies], axis=1)
STATE_DUMMY_COLS = list(state_dummies.columns)

print(f"\n✅ Carregados {len(datasets)} datasets")
print(f"   Dataset analítico principal: analise_compliance_municipio")
print(f"   Linhas: {len(df_analysis):,} municípios, {df_analysis['codigo_estado'].nunique()} estados, {df_analysis['codigo_regiao'].nunique()} regiões")
print(f"   Dummies regionais adicionadas ({len(REGION_DUMMY_COLS)}): {REGION_DUMMY_COLS}")
print(f"   Dummies de estado adicionadas ({len(STATE_DUMMY_COLS)}): primeiras 3 = {STATE_DUMMY_COLS[:3]} ... última = {STATE_DUMMY_COLS[-1]}")

## 2. Visão Geral do Dataset

In [ ]:
print("=" * 80)
print("DATASET ANALYSIS COMPLIANCE MUNICIPALITY")
print("=" * 80)
print(f"Dimensão: {df_analysis.shape}  (rows = municípios, cols = features + dummies)")
print(f"\nDtypes (apenas colunas não-dummy):")
print(df_analysis.drop(columns=REGION_DUMMY_COLS + STATE_DUMMY_COLS).dtypes)
print(f"\nUso de memória: {df_analysis.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

In [ ]:
# Pré-visualiza algumas colunas (ocultando as 32 dummies de região+estado).
preview_cols = [c for c in df_analysis.columns if c not in REGION_DUMMY_COLS + STATE_DUMMY_COLS]
df_analysis[preview_cols].head(10)

## 2.1 Inventário Completo dos Datasets da Camada Gold

Esta seção oferece uma visão abrangente de **TODOS os 6 datasets da camada Gold**.

| Dataset | Linhas | Granularidade | Propósito |
|---------|--------|---------------|-----------|
| `municipality_socioeconomic` | 5.570 | Município | Comparação censitária 2010→2022 com renda ajustada pela inflação |
| `state_summary` | 27 | Estado | Agregações estaduais (ponderadas pela população) |
| `sanctions_summary` | 3 | Tipo de registro | Sanções por registros CEIS/CNEP/CEPIM |
| `analysis_compliance` | 27 | Estado | Análise de compliance estadual (legado) |
| `analysis_compliance_municipality` | 5.570 | Município | **Dataset primário de análise** - granularidade principal |
| `consolidated_clustering` | 5.565 | Município | Pronto para ML com features normalizadas (z-score) |

### Campos de Renda Explicados

**Colunas de renda ajustadas pela inflação (IPCA, ano-base 2022 BRL):**
- `avg_income_real_2022_2022_brl` (`renda_media_real_2022_brl_2022`) - Renda 2022 em BRL de 2022 (ano-base)
- `avg_income_real_2010_2022_brl` (`renda_media_real_2010_brl_2022`) - Renda 2010 reexpressa em BRL de 2022
- `income_change_real_pct` (`mudanca_renda_real_pct`) - Variação real da renda (%) 2010→2022, ajustada pela inflação

**Colunas log-transformadas (para análise de regressão):**
- `log_population` (`log_populacao`) - Log natural da população (lida com assimetria)
- `log_income` (`log_renda`) - Log natural da renda real (base: avg_income_real_2022_2022_brl)
- `log_total_transfers` (`log_total_transferencias`) - Log1p das transferências federais (lida com zeros)


## Documentação dos Campos dos Datasets Gold

Documentação detalhada dos 6 datasets da camada Gold, incluindo tipos, descrições, método de cálculo e justificativa analítica.

> Nota: os nomes de colunas abaixo estão em inglês porque são **os nomes físicos no dataset original**. No notebook em português o loader `GoldDataLoaderPtBr` traduz para `codigo_municipio`, `populacao_2022`, etc. (ver `src/config/pt_br_translations.py`).

### Dataset: `municipality_socioeconomic` (pt-BR: `municipio_socioeconomico`)
**Agregação socioeconômica no nível municipal com comparação censitária 2010→2022 e renda ajustada pela inflação.**

| Nome do Campo | Tipo | Descrição | Cálculo | Justificativa |
|---|---|---|---|---|
| `municipality_code` | string | Identificador IBGE de 7 dígitos | De `dim_municipalities` | Chave estável para joins |
| `municipality_name` | string | Nome oficial do município | De `dim_municipalities` | Rótulo legível |
| `state_code` | string | Código IBGE de 2 dígitos do estado | De `dim_municipalities` | Chave de agregação estadual |
| `state_name` | string | Nome completo do estado | De `dim_municipalities` | Rótulo legível |
| `region_code` | int | Código IBGE da região (1-5) | De `dim_municipalities` | Agregação macrorregional |
| `region_name` | string | Nome da região | De `dim_municipalities` | Rótulo legível |
| `population_2010` / `population_2022` | int | População nos censos 2010 / 2022 | `fact_population` | Baseline + atual |
| `population_change_pct` | float | Variação populacional 2010→2022 (%) | `((pop_2022 - pop_2010) / pop_2010) * 100` | Indicador de crescimento |
| `literacy_rate_2010` / `literacy_rate_2022` | float | Taxa de alfabetização (%) | `fact_literacy` | Baseline + atual |
| `literacy_change_pp` | float | Variação em pontos percentuais | `literacy_2022 - literacy_2010` | Melhora educacional |
| `avg_income_2010` / `avg_income_2022` | float | Renda média nominal (BRL) | `fact_income` | Renda bruta |
| `avg_income_real_2010_2022_brl` | float | Renda 2010 em BRL de 2022 (IPCA) | `fact_income.avg_income_real_2022_brl` ano=2010 | Baseline real |
| `avg_income_real_2022_2022_brl` | float | Renda 2022 em BRL de 2022 | `fact_income.avg_income_real_2022_brl` ano=2022 | Renda real ano-base |
| `income_change_real_pct` | float | Variação real da renda (%) | Ajustada pela inflação | Mudança no poder de compra |
| `households_2010` / `households_2022` | int | Domicílios | `fact_sanitation` | Baseline + atual |

### Dataset: `state_summary` (`resumo_estado`)
**Agregações estaduais ponderadas pela população e resumo de sanções.**

27 linhas (26 estados + DF) com totais populacionais, médias ponderadas de alfabetização e renda, e contagens de sanções por registro (CEIS/CNEP/CEPIM).

### Dataset: `sanctions_summary` (`resumo_sancoes`)
**Visão agregada de sanções por tipo de registro (CEIS, CNEP, CEPIM).**

Três linhas, uma por tipo de registro, com totais, quantidade de municípios e estados envolvidos, média por município e participação relativa.

### Dataset: `analysis_compliance` (`analise_compliance`)
**Dataset legado de análise de compliance no nível estadual com dummies de região.**

27 linhas (N baixo). Contém transformações log (`log_population`, `log_income`) e binária `has_sanctions`. O dataset principal passou para granularidade municipal (ver abaixo).

### Dataset: `analysis_compliance_municipality` (`analise_compliance_municipio`) — DATASET PRIMÁRIO
**Principal dataset analítico no nível municipal. N = 5.570 — ordem de magnitude maior de poder estatístico que o rollup estadual.**

Inclui: identificadores IBGE, indicadores socioeconômicos 2022, renda real ajustada pelo IPCA, contagens de sanções (total + por registro), `sanctions_per_100k`, transferências federais e transformações log. É o dataset usado em todas as análises deste notebook e dos notebooks 02-05.

### Dataset: `consolidated_clustering` (`clustering_consolidado`)
**Dataset pronto para ML — features z-score normalizadas e análise de casos completos.**

5.565 linhas (5 a menos que o primário por remoção de dados faltantes). Contém valores brutos de ambos os anos censitários (2010 e 2022), versões `_norm` (z-score) para PCA/K-means, e transformações log. Usado nos notebooks 04 (clustering) e 05 (análise corrupção × IDH por cluster).

In [ ]:
# ============================================================
# DATASET 1: municipality_socioeconomic
# Granularidade: 1 linha por município (5,570 rows)
# Contém: AMBOS os anos censitários 2010 E 2022 com métricas de mudança
# ============================================================
print("=" * 80)
print("DATASET 1: municipality_socioeconomic (Ambos os Anos Censitários)")
print("=" * 80)
print(f"Dimensão: {df_muni.shape}")
print(f"\nColunas ({len(df_muni.columns)} total):")
print(df_muni.dtypes.to_string())
print(f"\n--- Colunas de renda (ajustadas pela inflação) ---")
income_cols = [c for c in df_muni.columns if 'income' in c.lower()]
for col in income_cols:
    print(f"  • {col}")
print(f"\n--- Amostra de dados (primeiras 5 linhas, colunas principais) ---")
key_cols = ['codigo_municipio', 'nome_municipio', 'populacao_2010', 'populacao_2022', 
            'renda_media_2010', 'renda_media_2022', 'renda_media_real_2010_brl_2022', 
            'renda_media_real_2022_brl_2022', 'mudanca_renda_real_pct']
display(df_muni[key_cols].head())

In [ ]:
# ============================================================
# DATASET 2: state_summary
# Granularidade: 1 linha por estado (27 rows = 26 states + DF)
# Contém: Agregações estaduais ponderadas pela população
# ============================================================
print("=" * 80)
print("DATASET 2: state_summary (Agregações no Nível Estadual)")
print("=" * 80)
print(f"Dimensão: {df_state.shape}")
print(f"\nColunas ({len(df_state.columns)} total):")
print(df_state.dtypes.to_string())
print(f"\n--- Colunas de renda (ambos os anos censitários) ---")
income_cols = [c for c in df_state.columns if 'income' in c.lower()]
for col in income_cols:
    print(f"  • {col}")
print(f"\n--- Amostra de dados (primeiros 5 estados) ---")
display(df_state.head())

In [ ]:
# ============================================================
# DATASET 3: sanctions_summary
# Granularidade: 1 linha por tipo de registro (3 rows: CEIS, CNEP, CEPIM)
# Contém: Contagens de sanções por registro com detalhamento PJ/PF
# ============================================================
print("=" * 80)
print("DATASET 3: sanctions_summary (Por Tipo de Registro)")
print("=" * 80)
print(f"Dimensão: {df_sanctions.shape}")
print(f"\nTodas as colunas:")
print(df_sanctions.dtypes.to_string())
print(f"\n--- Dataset completo (todas as 3 linhas) ---")
display(df_sanctions)

In [ ]:
# ============================================================
# DATASET 4: analysis_compliance (Estado-level)
# Granularidade: 1 linha por estado (27 rows)
# Contém: Análise legada no nível estadual com variáveis log-transformadas
# Nota: usa apenas dados de 2022 (único ano censitário)
# ============================================================
df_analysis_state = datasets.get('analise_compliance')

print("=" * 80)
print("DATASET 4: analysis_compliance (Nível Estadual (Legado))")
print("=" * 80)
print(f"Dimensão: {df_analysis_state.shape}")
print(f"\nColunas ({len(df_analysis_state.columns)} total):")
print(df_analysis_state.dtypes.to_string())
print(f"\n--- Colunas log-transformadas ---")
log_cols = [c for c in df_analysis_state.columns if c.startswith('log_')]
for col in log_cols:
    print(f"  • {col}")
print(f"\n--- Amostra de dados (primeiros 5 estados) ---")
display(df_analysis_state.head())

In [ ]:
# ============================================================
# DATASET 5: analysis_compliance_municipality
# Granularidade: 1 linha por município (5,570 rows)
# Contém: DATASET ANALÍTICO PRIMÁRIO (granularidade principal)
# Nota: usa apenas dados de 2022 + transferências federais
# ============================================================
print("=" * 80)
print("DATASET 5: analysis_compliance_municipality (PRIMÁRIO)")
print("=" * 80)
print(f"Dimensão: {df_analysis.shape} (inclui {len(REGION_DUMMY_COLS)} região + {len(STATE_DUMMY_COLS)} dummies de estado)")

# Show base columns (excluding dummies)
base_cols = [c for c in df_analysis.columns if c not in REGION_DUMMY_COLS + STATE_DUMMY_COLS]
print(f"\nColunas base ({len(base_cols)} total, excluindo dummies):")
for col in base_cols:
    print(f"  • {col} ({df_analysis[col].dtype})")

print(f"\n--- Colunas log-transformadas ---")
for col in ['log_populacao', 'log_renda', 'log_total_transferencias']:
    print(f"  • {col}")
print(f"\n--- Coluna de renda ajustada pelo IPCA ---")
print(f"  • avg_income_real_2022_2022_brl (ajustada pela inflação para BRL de 2022)")
print(f"\n--- Sample data (first 5 municípios, base columns) ---")
display(df_analysis[base_cols].head())

In [ ]:
# ============================================================
# DATASET 6: consolidated_clustering
# Granularidade: 1 linha por município (5.565 linhas, 5 a menos devido a dados faltantes)
# Contém: Dados prontos para ML com AMBOS 2010+2022 + features normalizadas z-score
# ============================================================
df_clustering = datasets.get('clustering_consolidado')

print("=" * 80)
print("DATASET 6: consolidated_clustering (Pronto para ML)")
print("=" * 80)
print(f"Dimensão: {df_clustering.shape}")

print(f"\n--- Colunas de valores brutos (ambos os anos censitários) ---")
raw_cols = [c for c in df_clustering.columns if not c.endswith('_norm') and 
            c not in ['codigo_municipio', 'nome_municipio', 'codigo_estado', 'state_abbrev', 
                     'nome_estado', 'codigo_regiao', 'nome_regiao']]
for col in raw_cols:
    print(f"  • {col}")

print(f"\n--- Colunas normalizadas (z-score, para ML) ---")
norm_cols = [c for c in df_clustering.columns if c.endswith('_norm')]
for col in norm_cols:
    print(f"  • {col}")

print(f"\n--- Colunas log (para regressão) ---")
log_cols_cluster = [c for c in df_clustering.columns if c.startswith('log_')]
for col in log_cols_cluster:
    print(f"  • {col}")

print(f"\n--- Amostra de dados: comparação 2010 vs 2022 ---")
compare_cols = ['nome_municipio', 'state_abbrev', 'populacao_2010', 'populacao_2022',
                'renda_media_real_2010_brl_2022', 'renda_media_real_2022_brl_2022',
                'mudanca_renda_real_pct']
display(df_clustering[compare_cols].head(10))

### Resumo dos Datasets: Qual Dataset Usar Quando?

| Objetivo da Análise | Dataset Recomendado | Por quê |
|---------------------|---------------------|---------|
| **Comparar mudança censitária 2010↔2022** | `municipality_socioeconomic` ou `consolidated_clustering` | Ambos contêm os dois anos censitários com ajuste pela inflação |
| **Análise de políticas no nível estadual** | `state_summary` | Agregações ponderadas pela população, estáveis |
| **Análise de registros de sanções** | `sanctions_summary` | Detalhamento CEIS/CNEP/CEPIM |
| **Análise principal da tese** | `analysis_compliance_municipality` | **Granularidade principal** — 5.570 municípios com transferências + sanções |
| **Modelagem de Regressão/ML** | `consolidated_clustering` | Limpo, normalizado, sem valores faltantes |
| **Tendências de compliance ao longo do tempo** | `analysis_compliance` (estado) ou versão municipal | Nível estadual para análise com N pequeno |

### Referência de Renda e Inflação

**Convenção de nomenclatura das colunas:** `avg_income_real_{value_year}_{base_year}_brl`
- `value_year`: o ano censitário em que a renda foi medida
- `base_year`: o ano-base da inflação (2022 = ano de referência do IPCA)

**Exemplos:**
- `avg_income_real_2022_2022_brl` = renda de 2022 em BRL de 2022 (nominal = real, ano-base)
- `avg_income_real_2010_2022_brl` = renda de 2010 reexpressa em BRL de 2022 (ajustada pela inflação)
- `income_change_real_pct` = variação real em % entre os dois (comparação ajustada pela inflação)

In [ ]:
df_analysis.info(verbose=False)

## 3. Estatísticas Descritivas

In [ ]:
# Descreve apenas as features analíticas, excluindo as 32 dummies one-hot
# (as dummies são resumidas separadamente abaixo).
numeric_cols = df_analysis.select_dtypes(include=[np.number]).columns
analytical_numeric = [c for c in numeric_cols if c not in REGION_DUMMY_COLS + STATE_DUMMY_COLS]
df_analysis[analytical_numeric].describe().T

### Resumo de Métricas Principais

In [ ]:
# Médias ponderadas pela população para taxas. Uma média simples entre os 5.570
# municípios trataria uma cidade de 500 habitantes e São Paulo igualmente,
# o que é estatisticamente enganoso para indicadores de taxa e valor médio.
_pop = df_analysis['populacao_2022']
_lit_mask = df_analysis['taxa_alfabetizacao_2022'].notna()
_inc_mask = df_analysis['renda_media_2022'].notna()

total_pop = int(_pop.sum())
total_sanc = int(df_analysis['num_sancoes'].sum())

summary = pd.DataFrame({
    'Total de Municípios': [len(df_analysis)],
    'Total de Estados': [int(df_analysis['codigo_estado'].nunique())],
    'Total de Regiões': [int(df_analysis['codigo_regiao'].nunique())],
    'População Total (2022)': [total_pop],
    'Total de Sanções': [total_sanc],
    'Sanções/100k (nacional, ponderada pela pop.)': [round(total_sanc / total_pop * 100_000, 2)],
    'Alfabetização % (ponderada pela pop.)': [round(np.average(df_analysis.loc[_lit_mask, 'taxa_alfabetizacao_2022'], weights=_pop[_lit_mask]), 2)],
    'Renda Média BRL (ponderada pela pop.)': [round(np.average(df_analysis.loc[_inc_mask, 'renda_media_2022'], weights=_pop[_inc_mask]), 2)],
})

summary.T

**Resumo das variáveis dummy.** Para as dummies one-hot de região e estado, a média equivale à proporção de municípios naquela região / estado (ex.: `df['is_region_1'].mean()` é a fração de munis na região Norte). Um resumo curto é mostrado abaixo para não inundar a tabela principal `.describe()` com 32 linhas extras.

In [ ]:
dummy_stats = pd.DataFrame({
    'Contagem (=1)': df_analysis[REGION_DUMMY_COLS + STATE_DUMMY_COLS].sum(),
    'Fração de munis %': (df_analysis[REGION_DUMMY_COLS + STATE_DUMMY_COLS].mean() * 100).round(2),
}).sort_values('Contagem (=1)', ascending=False)
print(f"Total de dummies: {len(dummy_stats)} (5 regiões + {len(STATE_DUMMY_COLS)} estados)")
dummy_stats.head(10)

## 4. Análise de Valores Ausentes

In [ ]:
missing = df_analysis.isnull().sum()
missing_pct = (missing / len(df_analysis)) * 100

missing_df = pd.DataFrame({
    'Qtd. Ausente': missing,
    '% Ausente': missing_pct.round(2),
}).sort_values('Qtd. Ausente', ascending=False)

missing_df[missing_df['Qtd. Ausente'] > 0]

## 5. Análise de Distribuição

### 5.1 Variável Alvo: Sanções por 100 mil

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df_analysis['sancoes_por_100k'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribuição de Sanções por 100 mil', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sanções por 100 mil habitantes')
axes[0].set_ylabel('Frequência')
axes[0].axvline(df_analysis['sancoes_por_100k'].mean(), color='red', linestyle='--', label='Média')
axes[0].axvline(df_analysis['sancoes_por_100k'].median(), color='green', linestyle='--', label='Mediana')
axes[0].legend()

axes[1].boxplot(df_analysis['sancoes_por_100k'])
axes[1].set_title('Boxplot: Sanções por 100 mil', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Sanções por 100 mil habitantes')

from scipy import stats
stats.probplot(df_analysis['sancoes_por_100k'], dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot: Sanções por 100 mil', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Assimetria (skewness): {df_analysis['sancoes_por_100k'].skew():.3f}")
print(f"Curtose: {df_analysis['sancoes_por_100k'].kurtosis():.3f}")


### 5.2 Indicadores Socioeconômicos

**Nota sobre a Transformação Logarítmica:** dados populacionais são tipicamente muito assimétricos à direita — alguns estados têm populações extremamente grandes enquanto a maioria é muito menor. A transformação logarítmica (log_population / log_populacao) trata isso:
1. **Normalizando a distribuição** — tornando-a mais simétrica para análise estatística válida
2. **Reduzindo a influência de valores extremos** — evitando que populações grandes afetem desproporcionalmente correlações e regressões
3. **Habilitando interpretação proporcional** — em modelos de regressão, mudanças representam efeitos proporcionais

Os histogramas abaixo comparam a distribuição populacional bruta (assimétrica) com a versão log-transformada (mais normal).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].hist(df_analysis['taxa_alfabetizacao_2022'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].set_title('Distribuição da Taxa de Alfabetização 2022 (por município)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Taxa de Alfabetização (%)')
axes[0, 0].set_ylabel('Frequência')

axes[0, 1].hist(df_analysis['renda_media_2022'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='lightgreen')
axes[0, 1].set_title('Distribuição da Renda Média 2022 (por município)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Renda Média (BRL)')
axes[0, 1].set_ylabel('Frequência')

axes[1, 0].hist(df_analysis['populacao_2022'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='salmon')
axes[1, 0].set_title('Distribuição da População 2022 (por município)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('População')
axes[1, 0].set_ylabel('Frequência')

axes[1, 1].hist(df_analysis['log_populacao'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='plum')
axes[1, 1].set_title('Log(População) Distribution (by municipality)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Log(População)')
axes[1, 1].set_ylabel('Frequência')

plt.tight_layout()
plt.show()

## 6. Análise Regional

In [ ]:
# Agregação regional a partir de dados em NÍVEL MUNICIPAL, usando
# estatísticas ponderadas. Um groupby('nome_regiao').agg('mean') trataria cada
# município igualmente, o que é estatisticamente enganoso: cidades pequenas
# dominariam a média para taxas e valores monetários.
# Para taxas e médias regionais, agregamos a partir dos totais
# e ponderamos pela população municipal.

def _region_rollup(g: pd.DataFrame) -> pd.Series:
    pop = g['populacao_2022']
    lit_mask = g['taxa_alfabetizacao_2022'].notna()
    inc_mask = g['renda_media_2022'].notna()
    return pd.Series({
        'N Municípios': len(g),
        'N Estados': g['codigo_estado'].nunique(),
        'População Total': int(pop.sum()),
        'Total de Sanções': int(g['num_sancoes'].sum()),
        'Sanções/100k (ponderada pela pop.)': round(g['num_sancoes'].sum() / pop.sum() * 100_000, 2),
        'Alfabetização % (ponderada pela pop.)': round(np.average(g.loc[lit_mask, 'taxa_alfabetizacao_2022'], weights=pop[lit_mask]), 2),
        'Renda Média BRL (ponderada pela pop.)': round(np.average(g.loc[inc_mask, 'renda_media_2022'], weights=pop[inc_mask]), 2),
    })

regional_summary = (
    df_analysis.groupby('nome_regiao', observed=True)
    .apply(_region_rollup)
)

regional_summary

In [ ]:
# Painel regional construído a partir dos dados municipais (taxas ponderadas pela pop.).
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Sanções por 100k por Região (ponderada pela pop.)',
                    'Taxa de Alfabetização por Região (ponderada pela pop.)',
                    'Renda Média por Região (ponderada pela pop.)',
                    'Total de Sanções por Região'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

regiões = regional_summary.reset_index()

fig.add_trace(go.Bar(x=regiões['nome_regiao'], y=regiões['Sanções/100k (ponderada pela pop.)'],
                     name='Sanctions/100k', marker_color='indianred'), row=1, col=1)
fig.add_trace(go.Bar(x=regiões['nome_regiao'], y=regiões['Alfabetização % (ponderada pela pop.)'],
                     name='Alfabetização %', marker_color='lightseagreen'), row=1, col=2)
fig.add_trace(go.Bar(x=regiões['nome_regiao'], y=regiões['Renda Média BRL (ponderada pela pop.)'],
                     name='Renda', marker_color='lightsalmon'), row=2, col=1)
fig.add_trace(go.Bar(x=regiões['nome_regiao'], y=regiões['Total de Sanções'],
                     name='Total de Sanções', marker_color='mediumpurple'), row=2, col=2)

fig.update_layout(height=800, showlegend=False, title_text="Painel Comparativo Regional (construído a partir de 5.570 municípios)")
fig.show()

## 7. Agregação por Estado e Extremos Municipais

Olhamos os dois extremos do espectro de granularidade:

1. **Agregação por estado** (ponderada pela população a partir dos 5.570 municípios) para um gráfico de barras estável e relevante para políticas.
2. **Top / bottom municípios** por `sanctions_per_100k` / `sancoes_por_100k`, que podem revelar outliers individuais que a agregação estadual esconde.

In [ ]:
# Top 10 e bottom 10 MUNICÍPIOS por sanções por 100 mil hab.
# OBS: taxas per capita para municípios muito pequenos podem ser instáveis
# (efeito denominador) -- usar com cautela.
cols = ['codigo_municipio', 'nome_municipio', 'nome_estado', 'nome_regiao',
        'populacao_2022', 'num_sancoes', 'sancoes_por_100k']

top_10_munis = df_analysis.nlargest(10, 'sancoes_por_100k')[cols]
bottom_10_munis = df_analysis.nsmallest(10, 'sancoes_por_100k')[cols]

print("TOP 10 MUNICÍPIOS - Maiores Sanções por 100 mil")
print("=" * 90)
print(top_10_munis.to_string(index=False))

print("\n\nBOTTOM 10 MUNICÍPIOS - Menores Sanções por 100 mil (entre munis com sanções > 0)")
print("=" * 90)
with_sanctions = df_analysis[df_analysis['num_sancoes'] > 0]
print(with_sanctions.nsmallest(10, 'sancoes_por_100k')[cols].to_string(index=False))

In [ ]:
# Agregação por estado a partir dos dados municipais (ponderada pela população).
# nome_estado é usado no eixo x (rótulo qualitativo); codigo_estado é apenas um id.
state_rollup = (
    df_analysis.groupby(['codigo_estado', 'nome_estado', 'nome_regiao'], observed=True)
    .apply(lambda g: pd.Series({
        'populacao': g['populacao_2022'].sum(),
        'num_sancoes': g['num_sancoes'].sum(),
        'sancoes_por_100k': g['num_sancoes'].sum() / g['populacao_2022'].sum() * 100_000,
    }))
    .reset_index()
    .sort_values('sancoes_por_100k', ascending=False)
)

fig = px.bar(state_rollup,
             x='nome_estado', y='sancoes_por_100k',
             color='nome_regiao',
             title='Sanções por 100 mil habitantes por Estado (agregação a partir de 5.570 municípios)',
             labels={'sancoes_por_100k': 'Sanções por 100 mil', 'nome_estado': 'Estado'},
             height=500)
fig.update_xaxes(tickangle=-45)
fig.show()

## 8. Análise de Registros de Sanções

In [ ]:
if df_sanctions is not None:
    print("Sanções por Tipo de Registro:")
    print("=" * 60)
    display(df_sanctions[['tipo_registro', 'total_sancoes', 'sancoes_pf', 'sancoes_pj', 'razao_pj_pct']])
    
    fig = px.pie(df_sanctions, values='total_sancoes', names='tipo_registro',
                 title='Distribuição de Sanções por Tipo de Registro')
    fig.show()


## 9. Mapa de Calor de Correlação (Prévia)

In [ ]:
# Matriz de correlação das principais features analíticas em nível municipal
# (5.570 observações em vez de 27 estados -- muito mais poder estatístico).
# We include the region dummies but NOT the 27 dummies de estado (would make the
# o heatmap ilegível). Features do lado das transferências também são incluídas,
# já que a pergunta do TCC liga transferências federais a resultados de compliance.

corr_cols = ['sancoes_por_100k', 'taxa_alfabetizacao_2022', 'renda_media_2022',
             'log_populacao', 'log_renda']
# Inclui log_total_transferencias se presente (novo no dataset muni)
if 'log_total_transferencias' in df_analysis.columns:
    corr_cols.append('log_total_transferencias')
corr_cols += REGION_DUMMY_COLS

# Converte para float64 (numpy) para que np.corrcoef / seaborn funcionem com pandas
# Int64/Float64 nullable + linhas que contêm NaN em qualquer coluna incluída.
corr_df = df_analysis[corr_cols].astype('Float64').astype(float)
corr_matrix = corr_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlação: Variáveis Principais (nível municipal, N=5.570)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Resumo dos Principais Achados

### 10.1 Qualidade dos Dados
- **Granularidade:** esta EDA roda em **nível municipal** (5.570 linhas, 1 por município brasileiro), ante os 27 registros da visão estadual anterior. Isso fornece ~200x mais poder estatístico para correlação/regressão a jusante.
- **Cobertura geográfica completa**: todos os 27 estados e 5 regiões representados.
- **Dados de sanções são densos**: `n_sanctions` / `num_sancoes` é não-nulo para os 5.570 municípios (zeros são reais, não ausentes).
- **Feature de taxa-por-transferências é esparsa**: `sanctions_per_million_brl_transfers` / `sancoes_por_milhao_brl_transferencias` é nulo em ~91,5% dos municípios (a maioria não tem registros de transferência federal no corte Gold atual). Usar com cautela em modelagens que dependam disso.

### 10.2 Padrões Regionais (ponderados pela população)
- Os números regionais são computados como `sum(sanctions) / sum(population) * 100_000` nos municípios de cada região — não como média não-ponderada de taxas por município — portanto não são dominados por municípios minúsculos.
- Ponderando corretamente, a ordem das regiões por sanções/100k é mais plana do que a visão estadual não-ponderada sugeria; veja a tabela de resumo regional acima para os valores reais no snapshot Gold atual.

### 10.3 Extremos de Estado e Município
- O gráfico de barras estadual agora é computado como **agregação a partir dos dados municipais** (ponderada pela população), então cada número estadual é consistente com os totais regionais.
- Os 10 municípios com maior `sanctions_per_100k` / `sancoes_por_100k` revelam outliers individuais escondidos pela agregação estadual. Taxas em municípios muito pequenos podem ser instáveis (efeito denominador) e devem ser interpretadas junto com `n_sanctions` / `num_sancoes` absoluto e `population_2022` / `populacao_2022`.

### 10.4 Variáveis dummy
- Região e estado são preservadas como colunas identificadoras (`state_code` / `codigo_estado`, etc.) E como dummies one-hot (`is_region_*`, `is_state_*`) para uso como features de regressão.
- A média de uma dummy equivale à proporção de municípios na categoria. Ver a tabela de resumo de dummies para as frações reais.

### 10.5 Correlações (nível municipal)
- Com N=5.570, os coeficientes de correlação no heatmap são muito mais confiáveis que no grão estadual de 27 observações. Inspecione a linha/coluna `sanctions_per_100k` / `sancoes_por_100k` no heatmap acima para os sinais bivariados mais fortes; a pergunta do TCC sobre transferências federais × resultados de compliance agora pode ser testada na unidade de observação onde a política efetivamente acontece (o município).